# Functions

In [1]:
def scopus_bib_to_df(file_path):
    """
    Reads a Scopus BibTeX file and converts it into a DataFrame with descriptive column names.

    Parameters:
    ----------
    file_path : str
        The path to the BibTeX file.

    Returns:
    -------
    pd.DataFrame
        DataFrame containing the processed BibTeX entries with descriptive column names.
    """
    import pandas as pd
    import re
    import unicodedata
    import bibtexparser

    try:
        with open(file_path, 'r', encoding='utf-8') as bibfile:
            bibtex_str = bibfile.read()

        # Preprocess the BibTeX string to sanitize field names
        def sanitize_field_names(bibtex_str):
            sanitized_lines = []
            for line in bibtex_str.splitlines():
                # Skip comment lines
                if line.strip().startswith('%'):
                    sanitized_lines.append(line)
                    continue
                # Find lines that look like field assignments
                if '=' in line:
                    parts = line.split('=', 1)
                    field_name = parts[0].strip()
                    field_value = parts[1]
                    # Normalize the field name to remove non-standard characters
                    field_name = unicodedata.normalize('NFKD', field_name)
                    field_name = field_name.encode('ascii', 'ignore').decode('ascii')
                    field_name = re.sub(r'\s+', '_', field_name)  # Replace whitespace with underscores
                    field_name = re.sub(r'[^a-zA-Z0-9_]', '', field_name)  # Remove any remaining non-alphanumeric characters
                    field_name = field_name.lower()
                    sanitized_line = f'{field_name} = {field_value}'
                    sanitized_lines.append(sanitized_line)
                else:
                    sanitized_lines.append(line)
            return '\n'.join(sanitized_lines)

        bibtex_str = sanitize_field_names(bibtex_str)

        # Parse the sanitized BibTeX entries
        bib_database = bibtexparser.loads(bibtex_str)
        entries = bib_database.entries

        entries_data = []

        for entry in entries:
            entry_data = {}
            try:
                for key in entry:
                    value = entry.get(key, '')
                    key_lower = key.lower()
                    # Ensure value is a string
                    if value is None:
                        value = ''
                    elif not isinstance(value, str):
                        value = str(value)
                    # Keep the 'url' field as is
                    if key_lower == 'url':
                        entry_data[key_lower] = value
                    else:
                        entry_data[key_lower] = value.upper()

                    # Process 'author' field
                    if key_lower == 'author':
                        authors_raw = value.replace('\n', ' ')
                        # Ensure authors_raw is a non-empty string
                        if isinstance(authors_raw, str) and authors_raw.strip():
                            # Split authors by ' and '
                            authors_list = re.split(r'\s+and\s+', authors_raw, flags=re.IGNORECASE)
                            processed_authors = []
                            for author in authors_list:
                                author = author.strip()
                                if ',' in author:
                                    # Format: 'Last, F.'
                                    last, first = [part.strip() for part in author.split(',', 1)]
                                    # Remove periods from initials
                                    first_initial = ''.join([name_part[0] for name_part in first.replace('.', '').split() if name_part])
                                else:
                                    # Format: 'First Last' or single name
                                    parts = author.split()
                                    if len(parts) >= 2:
                                        first_initial = parts[0][0]
                                        last = ' '.join(parts[1:])
                                    else:
                                        # Single name, take as last name
                                        first_initial = ''
                                        last = parts[0]
                                processed_author = f"{last.upper()} {first_initial.upper()}".strip()
                                processed_authors.append(processed_author)
                            entry_data[key_lower] = ';'.join(processed_authors)
                        else:
                            # Handle non-string or empty 'authors_raw'
                            entry_data[key_lower] = ''

                    # Process 'note' field to extract citation count
                    if key_lower == 'note':
                        note = value.upper()
                        match = re.search(r'CITED BY (\d+)', note)
                        citation_count = match.group(1) if match else ''
                        entry_data[key_lower] = citation_count

                    # Process 'affiliation' field to extract universities
                    if key_lower == 'affiliation':
                        affiliation = value.upper()
                        entry_data[key_lower] = affiliation

                        # Extract universities
                        university_pattern = r'([A-Z][A-Za-z\s]+(?:UNIVERSITY|COLLEGE|SCHOOL|INSTITUTE|ACADEMY|FACULTY|CENTER|DEPARTMENT)[A-Za-z\s]*)'
                        universities = re.findall(university_pattern, affiliation, re.IGNORECASE)
                        universities = [univ.strip().upper() for univ in universities]  # Clean up and uppercase university names
                        entry_data['au_un'] = '; '.join(universities)

                        # Get the first university
                        entry_data['au1_un'] = universities[0] if universities else ''

                entries_data.append(entry_data)

            except Exception as e:
                entry_id = entry.get('id', 'Unknown ID')
                print(f"Warning: Failed to process entry ID '{entry_id}': {e}")
                continue  # Skip to the next entry

        # Create the DataFrame
        df = pd.DataFrame(entries_data)

        # Define the mapping from BibTeX fields to descriptive DataFrame columns
        column_mapping = {
            'author': 'Authors',
            'author_keywords': 'Author Keywords',
            'keywords': 'Keywords Plus',
            'affiliation': 'Affiliations',
            'references': 'Cited References',
            'abbrev_source_title': 'Journal Abbreviation',
            'abstract': 'Abstract',
            'art_number': 'Article Number',
            'chemicals_cas': 'Chemicals CAS',
            'coden': 'CODEN',
            'correspondence_address1': 'Correspondence Address',
            'document_type': 'Document Type',
            'doi': 'DOI',
            'editor': 'Editor',
            'funding_details': 'Funding Details',
            'isbn': 'ISBN',
            'issn': 'ISSN',
            'journal': 'Journal',
            'language': 'Language',
            'note': 'Times Cited',
            'number': 'Issue',
            'page_count': 'Page Count',
            'pages': 'Pages',
            'publisher': 'Publisher',
            'pubmed_id': 'PubMed ID',
            'source': 'Database',
            'sponsors': 'Sponsors',
            'title': 'Title',
            'url': 'URL',
            'volume': 'Volume',
            'year': 'Year',
            'funding_text_1': 'Funding Text',
            # Additional mappings can be added here
        }

        # Rename columns according to the mapping
        df.rename(columns=column_mapping, inplace=True)

        # Rename 'au_un' and 'au1_un' to descriptive names
        df.rename(columns={'au_un': 'Author Universities', 'au1_un': 'First Author University'}, inplace=True)

        # Ensure all desired columns are present
        column_order = [
            'Authors', 'Author Keywords', 'Keywords Plus', 'Affiliations', 'Cited References', 'Journal Abbreviation',
            'Abstract', 'Article Number', 'Chemicals CAS', 'CODEN', 'Correspondence Address', 'Document Type', 'DOI',
            'Editor', 'Funding Details', 'ISBN', 'ISSN', 'Journal', 'Language', 'Times Cited', 'Issue', 'Page Count',
            'Pages', 'Publisher', 'PubMed ID', 'Database', 'Sponsors', 'Title', 'URL', 'Volume', 'Year', 'Funding Text',
            'Author Universities', 'First Author University', 'J9', 'SR_FULL', 'SR'
        ]

        for col in column_order:
            if col not in df.columns:
                df[col] = ''

        # Reorder the DataFrame columns
        df = df[column_order]

        # Create 'J9' by removing periods from 'Journal Abbreviation'
        df['J9'] = df['Journal Abbreviation'].str.replace('.', '', regex=False).fillna('')

        # Construct 'SR_FULL' and 'SR' columns using the renamed DataFrame columns
        def construct_sr_full(row):
            au = row['Authors']
            py = row['Year']
            so = row['Journal']
            if pd.notnull(au) and pd.notnull(py) and pd.notnull(so):
                if isinstance(au, str) and au:
                    first_author = au.split(';')[0]
                else:
                    first_author = ''
                return f"{first_author}, {py}, {so}"
            else:
                return ''

        def construct_sr(row):
            au = row['Authors']
            py = row['Year']
            j9 = row['J9']
            if pd.notnull(au) and pd.notnull(py) and pd.notnull(j9):
                if isinstance(au, str) and au:
                    first_author = au.split(';')[0]
                else:
                    first_author = ''
                return f"{first_author}, {py}, {j9}"
            else:
                return ''

        df['SR_FULL'] = df.apply(construct_sr_full, axis=1)
        df['SR'] = df.apply(construct_sr, axis=1)

        return df

    except FileNotFoundError:
        print(f"The file '{file_path}' was not found.")
        return None
    except UnicodeDecodeError:
        print(f"Encoding error while trying to read the file '{file_path}'.")
        return None
    except Exception as e:
        print(f"An unexpected error occurred: {e}")
        return None


# Data getting

In [2]:
scopus_df_1 = scopus_bib_to_df(r'C:\Users\PC-103\Documents\preprocessing\tests\files\scopus.bib')